# NYC Taxi Model Improvement

Purpose: improve the selected fare prediction model from the main analytics notebook using saved Delta tables and model artifacts. This notebook starts from the curated trip table, avoids rebuilding the full lakehouse, and focuses on faster model iteration.

## 1. Improvement Strategy

The v5 run showed that Model A is stable on robust labels but systematically underpredicts true fares by roughly **$9-10** across major test segments. The next modeling stage therefore focuses on **calibration**, **route-aware features**, and **temporal drift** rather than another broad full-pipeline rerun.

This notebook evaluates three layers:

- **Model A artifact baseline:** reload the selected ridge model from saved artifacts.
- **Model E calibration:** fit global and segment residual corrections on validation data only.
- **Model F enhanced ridge:** retrain a compact closed-form ridge model with route flags and year/month-index features.

In [ ]:
# Centralized imports for model improvement experiments.
import json
from typing import Optional

import numpy as np
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.conf.set("spark.sql.session.timeZone", "America/New_York")

## 2. Configuration

Use the same Unity Catalog objects and model artifact path as the main notebook. The default mode reads the curated table and saved Model A artifacts, then runs calibration and enhanced-feature experiments.

In [ ]:
# Project configuration.
CATALOG = "workspace"
SCHEMA = "bde"
VOLUME = "nyc_taxi"

TRIPS_TABLE = f"{CATALOG}.{SCHEMA}.taxi_trips_cleaned_borough"
MODEL_A_TEST_PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.model_a_test_predictions"
MODEL_E_TEST_PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.model_e_test_predictions"
MODEL_F_TEST_PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.model_f_test_predictions"
MODEL_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/models/model_a_ridge_v1"

OVERWRITE_TABLES = True
RUN_MODEL_A_BASELINE = True
RUN_BIAS_CALIBRATION = True
RUN_ENHANCED_RIDGE = True
SAVE_MODEL_PREDICTIONS = True
SEGMENT_MIN_ROWS = 10_000
SEGMENT_TOP_N = 25
EXTREME_AMOUNT_THRESHOLD = 1000.0
FARE_PER_MIN_THRESHOLD = 50.0
TAIL_REVIEW_TABLE = f"{CATALOG}.{SCHEMA}.model_e_tail_review"

write_mode = "overwrite" if OVERWRITE_TABLES else "errorifexists"

## 3. Shared Helpers

These helpers mirror the main notebook so this experiment can run independently inside Databricks. All preprocessing assets for Model A are loaded from the saved artifact directory.

In [ ]:
def predict_linear(
    df: DataFrame,
    feature_cols: list[str],
    weights,
    output_col: str,
) -> DataFrame:
    """Score a Spark DataFrame with an ordered linear model."""
    weight_values = [float(value) for value in np.asarray(weights).tolist()]
    prediction = None
    for weight, feature_col in zip(weight_values, feature_cols):
        term = F.lit(weight) * F.col(feature_col)
        prediction = term if prediction is None else prediction + term
    return df.withColumn(output_col, prediction)


def rmse(df: DataFrame, prediction_col: str, label_col: str) -> float:
    """Compute root mean squared error for Spark columns."""
    squared_error = F.pow(F.col(prediction_col) - F.col(label_col), 2)
    return float(
        df.select(squared_error.alias("squared_error"))
        .agg(F.sqrt(F.avg("squared_error")).alias("rmse"))
        .first()["rmse"]
    )


def mae(df: DataFrame, prediction_col: str, label_col: str) -> float:
    """Compute mean absolute error for Spark columns."""
    absolute_error = F.abs(F.col(prediction_col) - F.col(label_col))
    return float(
        df.select(absolute_error.alias("absolute_error"))
        .agg(F.avg("absolute_error").alias("mae"))
        .first()["mae"]
    )


def bias(df: DataFrame, prediction_col: str, label_col: str) -> float:
    """Compute average prediction minus actual label."""
    error = F.col(prediction_col) - F.col(label_col)
    return float(df.select(error.alias("error")).agg(F.avg("error").alias("bias")).first()["bias"])


def load_model_metadata(model_dir: str) -> dict:
    """Load Model A metadata from the configured artifact directory."""
    return json.loads(dbutils.fs.head(f"{model_dir}/metadata.json"))


def load_model_weights(model_dir: str, feature_order: list[str]) -> np.ndarray:
    """Load Model A weights in metadata feature order."""
    rows = spark.read.format("delta").load(f"{model_dir}/weights").collect()
    weight_by_feature = {row["feature"]: float(row["weight"]) for row in rows}
    return np.array([weight_by_feature[name] for name in feature_order], dtype=float)


def load_zscore_stats(model_dir: str) -> dict:
    """Load z-score statistics into the notebook stats dictionary format."""
    rows = spark.read.format("delta").load(f"{model_dir}/zscore_stats").collect()
    return {f"{row['col']}_{row['stat']}": float(row["value"]) for row in rows}


def load_target_encoding_maps(model_dir: str, columns: list[str]) -> dict[str, DataFrame]:
    """Load persisted target-encoding maps for configured columns."""
    return {
        col_name: spark.read.format("delta").load(f"{model_dir}/te_{col_name}")
        for col_name in columns
    }


def metric_row(model_name: str, df: DataFrame, prediction_col: str) -> tuple:
    """Return common model metrics on true and robust labels."""
    return (
        model_name,
        rmse(df, prediction_col, "total_amount"),
        mae(df, prediction_col, "total_amount"),
        bias(df, prediction_col, "total_amount"),
        rmse(df, prediction_col, "label_robust"),
    )

## 4. Load Curated Data

Reuse the curated Delta table created by the main notebook. The same time split is used so results compare directly with the v5 model output.

In [ ]:
# Build time-based train, validation, and test splits from the curated table.
source_df = spark.table(TRIPS_TABLE)
source_cols = set(source_df.columns)


def optional_col(col_name: str, dtype: str = "double"):
    """Return an optional source column with a typed null fallback."""
    if col_name in source_cols:
        return F.col(col_name).cast(dtype).alias(col_name)
    return F.lit(None).cast(dtype).alias(col_name)


base = (
    source_df
    .select(
        "total_amount",
        "trip_distance_km",
        "duration_min",
        F.coalesce(F.col("passenger_count"), F.lit(1)).alias("passenger_count"),
        "color",
        "pu_borough",
        "do_borough",
        "year",
        "month",
        F.dayofweek("pickup_ts").alias("dow"),
        F.hour("pickup_ts").alias("hour"),
        "pickup_ts",
        optional_col("payment_type", "int"),
        optional_col("RatecodeID", "int"),
        optional_col("fare_amount"),
        optional_col("extra"),
        optional_col("mta_tax"),
        optional_col("tip_amount"),
        optional_col("tolls_amount"),
        optional_col("improvement_surcharge"),
        optional_col("congestion_surcharge"),
        optional_col("airport_fee"),
        optional_col("PULocationID", "int"),
        optional_col("DOLocationID", "int"),
    )
    .where(
        "total_amount is not null "
        "and duration_min is not null "
        "and trip_distance_km is not null"
    )
)

train = base.where("pickup_ts < timestamp('2024-09-01')")
val = base.where(
    "pickup_ts >= timestamp('2024-09-01') "
    "and pickup_ts < timestamp('2024-10-01')"
)
test = base.where(
    "pickup_ts >= timestamp('2024-10-01') "
    "and pickup_ts < timestamp('2025-01-01')"
)

split_counts = (
    train.select(F.lit("train").alias("split"))
    .unionByName(val.select(F.lit("validation").alias("split")))
    .unionByName(test.select(F.lit("test").alias("split")))
    .groupBy("split")
    .count()
    .orderBy("split")
)
display(split_counts)

## 5. Rebuild Model A Scoring Features

Load the selected model artifacts from the main notebook, apply the same target encodings and standardization stats, and score validation/test rows with Model A. This creates a controlled baseline for improvement experiments.

In [ ]:
# Load Model A artifacts and apply its train-fitted preprocessing.
model_metadata = load_model_metadata(MODEL_DIR)
CAP = float(model_metadata.get("robust_cap", 500.0))
MODEL_A_FEATURE_COLS = model_metadata.get(
    "feature_order",
    ["bias", "dist", "dur", "pc", "hr", "mo", "dw", "color_te", "pu_te", "do_te"],
)
w_A = load_model_weights(MODEL_DIR, MODEL_A_FEATURE_COLS)
stats = load_zscore_stats(MODEL_DIR)

TE_COLS = ["color", "pu_borough", "do_borough"]
te_maps = load_target_encoding_maps(MODEL_DIR, TE_COLS)
y_global = float(model_metadata["y_global_for_TE"])


def clip_amount(col, cap: float):
    """Clip values above the train-fitted robust label cap."""
    return F.when(col > F.lit(cap), F.lit(cap)).otherwise(col)


def apply_target_encoding(df_in: DataFrame) -> DataFrame:
    """Join saved target encodings with global fallback values."""
    df = df_in
    for col_name in TE_COLS:
        encoded_col = f"{col_name}_te"
        df = df.join(te_maps[col_name], on=col_name, how="left")
        df = df.withColumn(encoded_col, F.coalesce(F.col(encoded_col), F.lit(y_global)))
    return df


def rename_borough_te_columns(df: DataFrame) -> DataFrame:
    """Shorten target-encoded borough feature names."""
    return (
        df.withColumnRenamed("pu_borough_te", "pu_te")
        .withColumnRenamed("do_borough_te", "do_te")
    )


def positive_stddev(col_name: str) -> float:
    """Return a non-zero training standard deviation for z-scoring."""
    std = stats[f"{col_name}_std"]
    return float(std) if std and std > 0 else 1.0


MODEL_A_NUM_COLS = [
    "trip_distance_km",
    "duration_min",
    "passenger_count",
    "hour",
    "month",
    "dow",
    "color_te",
    "pu_te",
    "do_te",
]


def apply_zscore(df: DataFrame, columns: list[str]) -> DataFrame:
    """Apply saved Model A z-score standardization."""
    out = df
    for col_name in columns:
        mean = stats[f"{col_name}_mean"]
        out = out.withColumn(
            f"{col_name}_z",
            (F.col(col_name) - F.lit(mean)) / F.lit(positive_stddev(col_name)),
        )
    return out


def add_model_a_features(df: DataFrame) -> DataFrame:
    """Materialize Model A feature columns and business context."""
    out = df.withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))
    out = rename_borough_te_columns(apply_target_encoding(out))
    out = apply_zscore(out, MODEL_A_NUM_COLS)
    return (
        out.withColumn("bias", F.lit(1.0))
        .withColumn("dist", F.col("trip_distance_km_z"))
        .withColumn("dur", F.col("duration_min_z"))
        .withColumn("pc", F.col("passenger_count_z"))
        .withColumn("hr", F.col("hour_z"))
        .withColumn("mo", F.col("month_z"))
        .withColumn("dw", F.col("dow_z"))
        .withColumn("color_te", F.col("color_te_z"))
        .withColumn("pu_te", F.col("pu_te_z"))
        .withColumn("do_te", F.col("do_te_z"))
    )

val_a_features = add_model_a_features(val)
test_a_features = add_model_a_features(test)

val_a = predict_linear(
    val_a_features,
    MODEL_A_FEATURE_COLS,
    w_A,
    output_col="prediction_A",
)
test_a = predict_linear(
    test_a_features,
    MODEL_A_FEATURE_COLS,
    w_A,
    output_col="prediction_A",
)

print(f"Model A validation RMSE: {rmse(val_a, 'prediction_A', 'total_amount'):.3f}")
print(f"Model A test RMSE:       {rmse(test_a, 'prediction_A', 'total_amount'):.3f}")
print(f"Model A test MAE:        {mae(test_a, 'prediction_A', 'total_amount'):.3f}")
print(f"Model A test bias:       {bias(test_a, 'prediction_A', 'total_amount'):.3f}")

## 6. Model E: Validation-Fitted Calibration

Model E corrects Model A predictions using residual patterns learned from the validation window only. This is intentionally lightweight: it tests whether the consistent underprediction can be reduced before we add more complex model features.

In [ ]:
# Fit global, color-level, and route-pair residual corrections on validation only.
def add_route_context(df: DataFrame) -> DataFrame:
    """Add route and duration segment labels for calibration and review."""
    return (
        df.withColumn(
            "route_pair",
            F.concat_ws(
                " -> ",
                F.coalesce(F.col("pu_borough"), F.lit("Unknown")),
                F.coalesce(F.col("do_borough"), F.lit("Unknown")),
            ),
        )
        .withColumn(
            "duration_bin",
            F.when(F.col("duration_min") < 5, F.lit("Under 5 Mins"))
            .when(F.col("duration_min") < 10, F.lit("5-10 Mins"))
            .when(F.col("duration_min") < 20, F.lit("10-20 Mins"))
            .when(F.col("duration_min") < 30, F.lit("20-30 Mins"))
            .when(F.col("duration_min") < 60, F.lit("30-60 Mins"))
            .otherwise(F.lit("At least 60 Mins")),
        )
    )

val_cal = add_route_context(val_a).withColumn(
    "residual_to_add",
    F.col("total_amount") - F.col("prediction_A"),
)
test_cal = add_route_context(test_a)

if RUN_BIAS_CALIBRATION:
    global_correction = float(
        val_cal.agg(F.avg("residual_to_add").alias("correction")).first()["correction"]
    )

    color_corrections = val_cal.groupBy("color").agg(
        F.avg("residual_to_add").alias("color_correction"),
        F.count(F.lit(1)).alias("color_n"),
    )

    route_corrections = (
        val_cal.groupBy("route_pair")
        .agg(
            F.avg("residual_to_add").alias("route_correction_raw"),
            F.count(F.lit(1)).alias("route_n"),
        )
        .where(F.col("route_n") >= F.lit(SEGMENT_MIN_ROWS))
        .withColumn(
            "route_correction",
            (F.col("route_correction_raw") * F.col("route_n") + F.lit(global_correction) * F.lit(SEGMENT_MIN_ROWS))
            / (F.col("route_n") + F.lit(SEGMENT_MIN_ROWS)),
        )
        .select("route_pair", "route_correction", "route_n")
    )

    test_e = (
        test_cal.join(color_corrections.select("color", "color_correction"), on="color", how="left")
        .join(route_corrections, on="route_pair", how="left")
        .withColumn("prediction_E_global", F.col("prediction_A") + F.lit(global_correction))
        .withColumn(
            "prediction_E_color",
            F.col("prediction_A") + F.coalesce(F.col("color_correction"), F.lit(global_correction)),
        )
        .withColumn(
            "prediction_E_route",
            F.col("prediction_A")
            + F.coalesce(
                F.col("route_correction"),
                F.col("color_correction"),
                F.lit(global_correction),
            ),
        )
    )

    print(f"Global validation correction: {global_correction:.3f}")
    display(route_corrections.orderBy(F.desc(F.abs("route_correction"))).limit(20))
else:
    test_e = test_cal
    print("Model E calibration skipped.")

## 7. Model F: Enhanced Ridge Features

Model F retrains a compact closed-form ridge model using the same robust target but adds route flags and temporal drift features. This keeps training Spark-friendly while testing the features suggested by the segment error analysis.

In [ ]:
# Add route-aware and temporal features for enhanced ridge training.
def add_improvement_features(df: DataFrame) -> DataFrame:
    """Add route flags and temporal features for Model F."""
    pu = F.coalesce(F.col("pu_borough"), F.lit("Unknown"))
    do = F.coalesce(F.col("do_borough"), F.lit("Unknown"))
    return (
        df.withColumn("label_robust", clip_amount(F.col("total_amount"), CAP))
        .withColumn("is_unknown_route", ((pu == "Unknown") | (do == "Unknown")).cast("double"))
        .withColumn("is_ewr_route", ((pu == "EWR") | (do == "EWR")).cast("double"))
        .withColumn("is_queens_manhattan", ((pu == "Queens") & (do == "Manhattan")).cast("double"))
        .withColumn("is_manhattan_queens", ((pu == "Manhattan") & (do == "Queens")).cast("double"))
        .withColumn("is_manhattan_internal", ((pu == "Manhattan") & (do == "Manhattan")).cast("double"))
        .withColumn("month_index", (F.col("year") - F.lit(2009)) * F.lit(12) + F.col("month"))
    )

train_imp = add_improvement_features(train)
val_imp = add_improvement_features(val)
test_imp = add_improvement_features(test)

# Reuse saved borough target encodings, then fit only new numeric stats from train.
train_imp = rename_borough_te_columns(apply_target_encoding(train_imp))
val_imp = rename_borough_te_columns(apply_target_encoding(val_imp))
test_imp = rename_borough_te_columns(apply_target_encoding(test_imp))

MODEL_F_NUM_COLS = [
    "trip_distance_km",
    "duration_min",
    "passenger_count",
    "hour",
    "month",
    "dow",
    "month_index",
    "color_te",
    "pu_te",
    "do_te",
]

model_f_stats = (
    train_imp.agg(
        *[F.avg(col_name).alias(f"{col_name}_mean") for col_name in MODEL_F_NUM_COLS],
        *[F.stddev_pop(col_name).alias(f"{col_name}_std") for col_name in MODEL_F_NUM_COLS],
    )
    .collect()[0]
    .asDict()
)


def model_f_std(col_name: str) -> float:
    """Return a non-zero train-fitted standard deviation for Model F."""
    std = model_f_stats[f"{col_name}_std"]
    return float(std) if std and std > 0 else 1.0


def apply_model_f_zscore(df: DataFrame) -> DataFrame:
    """Apply Model F train-fitted z-score standardization."""
    out = df
    for col_name in MODEL_F_NUM_COLS:
        mean = model_f_stats[f"{col_name}_mean"]
        out = out.withColumn(
            f"{col_name}_z",
            (F.col(col_name) - F.lit(mean)) / F.lit(model_f_std(col_name)),
        )
    return out

train_fz = apply_model_f_zscore(train_imp)
val_fz = apply_model_f_zscore(val_imp)
test_fz = apply_model_f_zscore(test_imp)

MODEL_F_FEATURES = [
    "bias",
    "dist",
    "dur",
    "pc",
    "hr",
    "mo",
    "dw",
    "month_idx",
    "color_te_f",
    "pu_te_f",
    "do_te_f",
    "unknown_route",
    "ewr_route",
    "queens_manhattan",
    "manhattan_queens",
    "manhattan_internal",
]


def with_model_f_features(df: DataFrame) -> DataFrame:
    """Materialize ordered Model F numeric feature columns."""
    return (
        df.withColumn("bias", F.lit(1.0))
        .withColumn("dist", F.col("trip_distance_km_z"))
        .withColumn("dur", F.col("duration_min_z"))
        .withColumn("pc", F.col("passenger_count_z"))
        .withColumn("hr", F.col("hour_z"))
        .withColumn("mo", F.col("month_z"))
        .withColumn("dw", F.col("dow_z"))
        .withColumn("month_idx", F.col("month_index_z"))
        .withColumn("color_te_f", F.col("color_te_z"))
        .withColumn("pu_te_f", F.col("pu_te_z"))
        .withColumn("do_te_f", F.col("do_te_z"))
        .withColumn("unknown_route", F.col("is_unknown_route"))
        .withColumn("ewr_route", F.col("is_ewr_route"))
        .withColumn("queens_manhattan", F.col("is_queens_manhattan"))
        .withColumn("manhattan_queens", F.col("is_manhattan_queens"))
        .withColumn("manhattan_internal", F.col("is_manhattan_internal"))
    )

train_ff = with_model_f_features(train_fz)
val_ff = with_model_f_features(val_fz)
test_ff = with_model_f_features(test_fz)

print("Model F feature dimension:", len(MODEL_F_FEATURES))

In [ ]:
# Train Model F with closed-form ridge regression on robust labels.
def fit_closed_form_ridge(
    df_train: DataFrame,
    feature_cols: list[str],
    label_col: str,
    lam: float = 1e-3,
) -> np.ndarray:
    """Fit a compact ridge model through aggregated normal equations."""
    p = len(feature_cols)
    agg_exprs = []
    for i, ci in enumerate(feature_cols):
        for j, cj in enumerate(feature_cols):
            if j >= i:
                agg_exprs.append(F.sum(F.col(ci) * F.col(cj)).alias(f"G_{i}_{j}"))

    b_exprs = [
        F.sum(F.col(ci) * F.col(label_col)).alias(f"b_{i}")
        for i, ci in enumerate(feature_cols)
    ]
    row = df_train.agg(*(agg_exprs + b_exprs)).collect()[0].asDict()

    gram = np.zeros((p, p), dtype=float)
    for i in range(p):
        for j in range(i, p):
            gram[i, j] = float(row[f"G_{i}_{j}"])
            gram[j, i] = gram[i, j]

    rhs = np.array([float(row[f"b_{i}"]) for i in range(p)], dtype=float)
    return np.linalg.solve(gram + lam * np.eye(p), rhs)


if RUN_ENHANCED_RIDGE:
    w_F = fit_closed_form_ridge(train_ff, MODEL_F_FEATURES, "label_robust")
    val_f_model = predict_linear(val_ff, MODEL_F_FEATURES, w_F, "prediction_F")
    test_f_model = predict_linear(test_ff, MODEL_F_FEATURES, w_F, "prediction_F")

    print(f"Model F validation RMSE: {rmse(val_f_model, 'prediction_F', 'total_amount'):.3f}")
    print(f"Model F test RMSE:       {rmse(test_f_model, 'prediction_F', 'total_amount'):.3f}")
    print(f"Model F test MAE:        {mae(test_f_model, 'prediction_F', 'total_amount'):.3f}")
    print(f"Model F test bias:       {bias(test_f_model, 'prediction_F', 'total_amount'):.3f}")
else:
    test_f_model = None
    print("Model F training skipped.")

## 8. Model Comparison

Compare Model A, calibrated Model E variants, and enhanced Model F on the October-December 2024 test window. Use RMSE for compatibility with the original objective and MAE/bias to understand practical error behavior.

In [ ]:
# Compare all available test predictions.
comparison_rows = [metric_row("Model_A_artifact", test_a, "prediction_A")]

if RUN_BIAS_CALIBRATION:
    comparison_rows.extend(
        [
            metric_row("Model_E_global_calibration", test_e, "prediction_E_global"),
            metric_row("Model_E_color_calibration", test_e, "prediction_E_color"),
            metric_row("Model_E_route_calibration", test_e, "prediction_E_route"),
        ]
    )

if RUN_ENHANCED_RIDGE and test_f_model is not None:
    comparison_rows.append(metric_row("Model_F_enhanced_ridge", test_f_model, "prediction_F"))

schema = T.StructType(
    [
        T.StructField("model", T.StringType(), False),
        T.StructField("rmse_true", T.DoubleType(), False),
        T.StructField("mae_true", T.DoubleType(), False),
        T.StructField("bias_true", T.DoubleType(), False),
        T.StructField("rmse_robust", T.DoubleType(), False),
    ]
)

comparison_df = (
    spark.createDataFrame(comparison_rows, schema)
    .select(
        "model",
        F.round("rmse_true", 3).alias("rmse_true"),
        F.round("mae_true", 3).alias("mae_true"),
        F.round("bias_true", 3).alias("bias_true"),
        F.round("rmse_robust", 3).alias("rmse_robust"),
    )
    .orderBy("rmse_true")
)

display(comparison_df)

## 9. Segment Diagnostics

Review the best available improved prediction by the same business segments used in the main notebook. This keeps improvement work anchored to route, month, duration, and taxi-color behavior rather than only aggregate scores.

In [ ]:
# Choose the preferred improved prediction for segment diagnostics.
if RUN_BIAS_CALIBRATION:
    diagnostic_df = test_e.withColumn("prediction_best", F.col("prediction_E_route"))
    diagnostic_model_name = "Model_E_route_calibration"
elif RUN_ENHANCED_RIDGE and test_f_model is not None:
    diagnostic_df = add_route_context(test_f_model).withColumn("prediction_best", F.col("prediction_F"))
    diagnostic_model_name = "Model_F_enhanced_ridge"
else:
    diagnostic_df = add_route_context(test_a).withColumn("prediction_best", F.col("prediction_A"))
    diagnostic_model_name = "Model_A_artifact"


def segment_error_metrics(
    df: DataFrame,
    segment_cols: list[str],
    min_rows: int = SEGMENT_MIN_ROWS,
) -> DataFrame:
    """Summarize model error metrics by one or more segment columns."""
    scored = df.withColumn("error", F.col("prediction_best") - F.col("total_amount"))
    return (
        scored.groupBy(*segment_cols)
        .agg(
            F.count(F.lit(1)).alias("trip_count"),
            F.avg("total_amount").alias("avg_actual"),
            F.avg("prediction_best").alias("avg_predicted"),
            F.avg("error").alias("bias"),
            F.avg(F.abs(F.col("error"))).alias("mae"),
            F.sqrt(F.avg(F.pow(F.col("error"), 2))).alias("rmse"),
        )
        .where(F.col("trip_count") >= F.lit(min_rows))
        .select(
            *segment_cols,
            "trip_count",
            F.round("avg_actual", 2).alias("avg_actual"),
            F.round("avg_predicted", 2).alias("avg_predicted"),
            F.round("bias", 2).alias("bias"),
            F.round("mae", 2).alias("mae"),
            F.round("rmse", 2).alias("rmse"),
        )
    )

print(f"Segment diagnostics for: {diagnostic_model_name}")

print("Error by taxi color")
display(segment_error_metrics(diagnostic_df, ["color"]).orderBy("color"))

print("Error by month")
display(segment_error_metrics(diagnostic_df, ["month"]).orderBy("month"))

print("Error by duration bin")
display(
    segment_error_metrics(diagnostic_df, ["duration_bin"])
    .orderBy(
        F.when(F.col("duration_bin") == "Under 5 Mins", 1)
        .when(F.col("duration_bin") == "5-10 Mins", 2)
        .when(F.col("duration_bin") == "10-20 Mins", 3)
        .when(F.col("duration_bin") == "20-30 Mins", 4)
        .when(F.col("duration_bin") == "30-60 Mins", 5)
        .otherwise(6)
    )
)

print("Highest-error route pairs")
display(
    segment_error_metrics(diagnostic_df, ["route_pair"])
    .orderBy(F.desc("rmse"), F.desc("trip_count"))
    .limit(SEGMENT_TOP_N)
)

## 10. Tail Error Analysis

Model E removes most systematic bias, so the remaining true-RMSE problem is dominated by high-impact tail rows. This section inspects the rows and segments contributing the largest squared error, especially November, 10-20 minute trips, Manhattan internal trips, EWR, and Unknown borough routes.


In [ ]:
# Inspect tail errors that dominate true-label RMSE.
tail_scored = (
    diagnostic_df
    .withColumn("error", F.col("prediction_best") - F.col("total_amount"))
    .withColumn("abs_error", F.abs(F.col("error")))
    .withColumn("squared_error", F.pow(F.col("error"), 2))
    .withColumn(
        "fare_per_min",
        F.when(F.col("duration_min") > 0, F.col("total_amount") / F.col("duration_min")),
    )
    .withColumn(
        "is_priority_tail_segment",
        (
            (F.col("month") == 11)
            | (F.col("duration_bin") == "10-20 Mins")
            | (F.col("route_pair") == "Manhattan -> Manhattan")
            | F.col("route_pair").contains("EWR")
            | F.col("route_pair").contains("Unknown")
        ),
    )
)

print("Error distribution for selected improved model")
display(
    tail_scored.selectExpr(
        "percentile_approx(abs_error, array(0.5, 0.9, 0.95, 0.99, 0.999), 10000) AS abs_error_quantiles",
        "percentile_approx(total_amount, array(0.5, 0.9, 0.95, 0.99, 0.999), 10000) AS total_amount_quantiles",
        "MAX(abs_error) AS max_abs_error",
        "MAX(total_amount) AS max_total_amount"
    )
)

print("Highest absolute-error rows")
display(
    tail_scored.select(
        "month",
        "pickup_ts",
        "color",
        "route_pair",
        "duration_bin",
        F.round("duration_min", 2).alias("duration_min"),
        F.round("trip_distance_km", 2).alias("trip_distance_km"),
        "payment_type",
        "RatecodeID",
        F.round("fare_amount", 2).alias("fare_amount"),
        F.round("tip_amount", 2).alias("tip_amount"),
        F.round("tolls_amount", 2).alias("tolls_amount"),
        F.round("airport_fee", 2).alias("airport_fee"),
        F.round("total_amount", 2).alias("total_amount"),
        F.round("prediction_best", 2).alias("prediction_best"),
        F.round("error", 2).alias("error"),
        F.round("abs_error", 2).alias("abs_error"),
        F.round("fare_per_min", 2).alias("fare_per_min"),
    )
    .orderBy(F.desc("abs_error"))
    .limit(50)
)


In [ ]:
# Quantify which business segments drive squared error.
def tail_segment_summary(segment_cols: list[str]) -> DataFrame:
    """Summarize tail-error contribution by segment."""
    total_sse = tail_scored.agg(F.sum("squared_error").alias("sse")).first()["sse"]
    return (
        tail_scored.groupBy(*segment_cols)
        .agg(
            F.count(F.lit(1)).alias("trip_count"),
            F.avg("total_amount").alias("avg_actual"),
            F.avg("prediction_best").alias("avg_predicted"),
            F.avg("error").alias("bias"),
            F.avg("abs_error").alias("mae"),
            F.sqrt(F.avg("squared_error")).alias("rmse"),
            F.sum("squared_error").alias("segment_sse"),
            F.sum(F.when(F.col("abs_error") >= 100, 1).otherwise(0)).alias("abs_error_100_plus"),
        )
        .where(F.col("trip_count") >= F.lit(SEGMENT_MIN_ROWS))
        .withColumn("sse_share_pct", F.col("segment_sse") / F.lit(total_sse) * F.lit(100.0))
        .select(
            *segment_cols,
            "trip_count",
            F.round("avg_actual", 2).alias("avg_actual"),
            F.round("avg_predicted", 2).alias("avg_predicted"),
            F.round("bias", 2).alias("bias"),
            F.round("mae", 2).alias("mae"),
            F.round("rmse", 2).alias("rmse"),
            "abs_error_100_plus",
            F.round("sse_share_pct", 2).alias("sse_share_pct"),
        )
    )

print("Squared-error contribution by month")
display(tail_segment_summary(["month"]).orderBy(F.desc("sse_share_pct")))

print("Squared-error contribution by duration bin")
display(tail_segment_summary(["duration_bin"]).orderBy(F.desc("sse_share_pct")))

print("Squared-error contribution by route pair")
display(
    tail_segment_summary(["route_pair"])
    .orderBy(F.desc("sse_share_pct"), F.desc("rmse"))
    .limit(SEGMENT_TOP_N)
)


In [ ]:
# Inspect likely explanations for the remaining high-error tail.
print("Tail-risk summary by fare and route flags")
display(
    tail_scored.select(
        F.when(F.col("route_pair").contains("EWR"), F.lit("EWR route"))
        .when(F.col("route_pair").contains("Unknown"), F.lit("Unknown route"))
        .when(F.col("route_pair") == "Manhattan -> Manhattan", F.lit("Manhattan internal"))
        .otherwise(F.lit("Other route"))
        .alias("route_flag"),
        F.when(F.col("total_amount") >= 200, F.lit("$200+"))
        .when(F.col("total_amount") >= 135.09, F.lit("robust-cap+"))
        .when(F.col("total_amount") >= 100, F.lit("$100-$135"))
        .otherwise(F.lit("under $100"))
        .alias("fare_band"),
        "payment_type",
        "RatecodeID",
        "abs_error",
        "squared_error",
        "total_amount",
        "prediction_best",
    )
    .groupBy("route_flag", "fare_band", "payment_type", "RatecodeID")
    .agg(
        F.count(F.lit(1)).alias("trip_count"),
        F.avg("total_amount").alias("avg_actual"),
        F.avg("prediction_best").alias("avg_predicted"),
        F.avg("abs_error").alias("mae"),
        F.sqrt(F.avg("squared_error")).alias("rmse"),
    )
    .where(F.col("trip_count") >= F.lit(100))
    .select(
        "route_flag",
        "fare_band",
        "payment_type",
        "RatecodeID",
        "trip_count",
        F.round("avg_actual", 2).alias("avg_actual"),
        F.round("avg_predicted", 2).alias("avg_predicted"),
        F.round("mae", 2).alias("mae"),
        F.round("rmse", 2).alias("rmse"),
    )
    .orderBy(F.desc("rmse"), F.desc("trip_count"))
    .limit(50)
)


## 11. Operational Metric View

Keep two model-quality views: the **full raw metric** for transparency and an **operational metric** that excludes clearly extreme fare anomalies. This makes the remaining RMSE interpretable without pretending the outliers do not exist.


In [ ]:
# Flag extreme tail rows for operational reporting.
EXTREME_AMOUNT_THRESHOLD = globals().get("EXTREME_AMOUNT_THRESHOLD", 1000.0)
FARE_PER_MIN_THRESHOLD = globals().get("FARE_PER_MIN_THRESHOLD", 50.0)
TAIL_REVIEW_TABLE = globals().get(
    "TAIL_REVIEW_TABLE",
    f"{CATALOG}.{SCHEMA}.model_e_tail_review",
)

tail_flagged = (
    tail_scored
    .withColumn(
        "is_extreme_amount",
        F.col("total_amount") > F.lit(EXTREME_AMOUNT_THRESHOLD),
    )
    .withColumn(
        "is_extreme_fare_per_min",
        F.col("fare_per_min") > F.lit(FARE_PER_MIN_THRESHOLD),
    )
    .withColumn(
        "is_suspicious_manhattan_internal",
        (
            (F.col("route_pair") == "Manhattan -> Manhattan")
            & (F.col("total_amount") >= F.lit(100.0))
            & (F.col("duration_min") < F.lit(30.0))
        ),
    )
    .withColumn(
        "is_november_10_20_tail",
        (
            (F.col("month") == 11)
            & (F.col("duration_bin") == "10-20 Mins")
            & (F.col("abs_error") >= F.lit(100.0))
        ),
    )
    .withColumn(
        "is_operational_anomaly",
        F.col("is_extreme_amount")
        | F.col("is_extreme_fare_per_min")
        | F.col("is_suspicious_manhattan_internal")
        | F.col("is_november_10_20_tail"),
    )
)

print("Operational anomaly counts")
display(
    tail_flagged.agg(
        F.count(F.lit(1)).alias("total_rows"),
        F.sum(F.col("is_operational_anomaly").cast("int")).alias("anomaly_rows"),
        F.sum(F.col("is_extreme_amount").cast("int")).alias("extreme_amount_rows"),
        F.sum(F.col("is_extreme_fare_per_min").cast("int")).alias("extreme_fare_per_min_rows"),
        F.sum(F.col("is_suspicious_manhattan_internal").cast("int")).alias("suspicious_manhattan_internal_rows"),
        F.sum(F.col("is_november_10_20_tail").cast("int")).alias("november_10_20_tail_rows"),
    )
    .withColumn("anomaly_pct", F.round(F.col("anomaly_rows") / F.col("total_rows") * 100, 4))
)

print("Top anomaly reasons")
display(
    tail_flagged.select(
        F.explode(
            F.array(
                F.when(F.col("is_extreme_amount"), F.lit("total_amount > threshold")),
                F.when(F.col("is_extreme_fare_per_min"), F.lit("fare_per_min > threshold")),
                F.when(F.col("is_suspicious_manhattan_internal"), F.lit("high-fare Manhattan internal")),
                F.when(F.col("is_november_10_20_tail"), F.lit("November 10-20 min abs_error >= 100")),
            )
        ).alias("anomaly_reason")
    )
    .where(F.col("anomaly_reason").isNotNull())
    .groupBy("anomaly_reason")
    .count()
    .orderBy(F.desc("count"))
)


In [ ]:
# Compare transparent full metrics with operational filtered/capped views.
def scored_metrics(label: str, df: DataFrame, prediction_col: str, target_col: str) -> tuple:
    """Return row count and core metrics for one reporting view."""
    return (
        label,
        int(df.count()),
        rmse(df, prediction_col, target_col),
        mae(df, prediction_col, target_col),
        bias(df, prediction_col, target_col),
    )

full_metric_df = tail_flagged.withColumn("target_capped_1000", F.least(F.col("total_amount"), F.lit(EXTREME_AMOUNT_THRESHOLD)))
operational_metric_df = full_metric_df.where(~F.col("is_operational_anomaly"))

reporting_rows = [
    scored_metrics("Full raw metric", full_metric_df, "prediction_best", "total_amount"),
    scored_metrics("Full capped at 1000", full_metric_df, "prediction_best", "target_capped_1000"),
    scored_metrics("Operational filtered", operational_metric_df, "prediction_best", "total_amount"),
]

reporting_schema = T.StructType(
    [
        T.StructField("metric_view", T.StringType(), False),
        T.StructField("row_count", T.LongType(), False),
        T.StructField("rmse", T.DoubleType(), False),
        T.StructField("mae", T.DoubleType(), False),
        T.StructField("bias", T.DoubleType(), False),
    ]
)

reporting_metrics = (
    spark.createDataFrame(reporting_rows, reporting_schema)
    .select(
        "metric_view",
        "row_count",
        F.round("rmse", 3).alias("rmse"),
        F.round("mae", 3).alias("mae"),
        F.round("bias", 3).alias("bias"),
    )
)

display(reporting_metrics)


In [ ]:
# Persist a compact tail-review table for audit and dashboard follow-up.
if SAVE_MODEL_PREDICTIONS:
    (
        tail_flagged.select(
            "month",
            "pickup_ts",
            "color",
            "route_pair",
            "duration_bin",
            "duration_min",
            "trip_distance_km",
            "payment_type",
            "RatecodeID",
            "fare_amount",
            "tip_amount",
            "tolls_amount",
            "airport_fee",
            "total_amount",
            "prediction_best",
            "error",
            "abs_error",
            "fare_per_min",
            "is_extreme_amount",
            "is_extreme_fare_per_min",
            "is_suspicious_manhattan_internal",
            "is_november_10_20_tail",
            "is_operational_anomaly",
        )
        .where(F.col("is_operational_anomaly") | (F.col("abs_error") >= F.lit(100.0)))
        .write.mode(write_mode)
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(TAIL_REVIEW_TABLE)
    )
    print(f"Saved tail review rows to: {TAIL_REVIEW_TABLE}")


## 12. Persist Improved Predictions

Persist the calibrated and enhanced-model test predictions as lightweight Delta tables. These tables are useful for dashboards, report screenshots, and future monitoring without rerunning training.

In [ ]:
# Persist improved predictions for downstream review.
if SAVE_MODEL_PREDICTIONS and RUN_BIAS_CALIBRATION:
    (
        test_e.select(
            "color",
            "pu_borough",
            "do_borough",
            "month",
            "hour",
            "duration_min",
            "duration_bin",
            "route_pair",
            "total_amount",
            "label_robust",
            "prediction_A",
            "prediction_E_global",
            "prediction_E_color",
            "prediction_E_route",
        )
        .write.mode(write_mode)
        .format("delta")
        .saveAsTable(MODEL_E_TEST_PREDICTIONS_TABLE)
    )
    print(f"Saved Model E predictions to: {MODEL_E_TEST_PREDICTIONS_TABLE}")

if SAVE_MODEL_PREDICTIONS and RUN_ENHANCED_RIDGE and test_f_model is not None:
    (
        add_route_context(test_f_model)
        .select(
            "color",
            "pu_borough",
            "do_borough",
            "month",
            "hour",
            "duration_min",
            "duration_bin",
            "route_pair",
            "total_amount",
            "label_robust",
            "prediction_F",
        )
        .write.mode(write_mode)
        .format("delta")
        .saveAsTable(MODEL_F_TEST_PREDICTIONS_TABLE)
    )
    print(f"Saved Model F predictions to: {MODEL_F_TEST_PREDICTIONS_TABLE}")

## 13. Promotion Criteria

Promote an improvement back into the main notebook only when it improves more than one metric and does not create worse segment behavior. The strongest candidate should reduce **bias**, preserve or improve **MAE**, and reduce either true-label or robust **RMSE** on the October-December 2024 test window. If true RMSE remains high while MAE is low, use the tail-error tables to decide whether to add new features or tighten data quality rules for extreme fare cases.